# HF-RISK Colab Runner

Run this notebook with the **Google Colab VS Code extension**. The normal VS Code PowerShell terminal still uses your laptop; these notebook cells use the selected Colab runtime when the kernel/runtime is set to Colab.

## No Google Drive workflow

If Colab cannot see your local project files, upload `hf_risk_colab_bundle.zip` to the Colab server using the VS Code Colab extension. In VS Code: right-click the ZIP or use the Command Palette command `Colab: Upload to Colab` / `Upload to Colab`. The next cell will unzip it into `/content/MSc Project`.

Pipeline order: `02_preprocessing.py -> 03_modelling.py -> 04_survival.py -> 05_shap.py -> 08_external_validation_mimic.py -> 10_advanced_evaluation.py`.


In [8]:
# 1) Confirm this is running on Colab, not local Windows.
import os, sys, platform

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())
print('Current folder:', os.getcwd())

runtime_os = os.environ.get('OS', os.name).lower()
if runtime_os == 'nt' or runtime_os.startswith('windows'):
    print('WARNING: This kernel is Windows/local. Select a Colab runtime in VS Code before running the heavy scripts.')
else:
    print('OK: Non-Windows runtime detected. This should be Colab/Linux if you selected Colab.')

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Current folder: /content
OK: Non-Windows runtime detected. This should be Colab/Linux if you selected Colab.


In [7]:
# 2) Locate the MSc Project folder in the Colab runtime.
# If you uploaded hf_risk_colab_bundle.zip via the Colab VS Code extension, this cell unzips it first.
from pathlib import Path
import os, zipfile

def looks_like_project(folder: Path) -> bool:
    return (folder / '02_preprocessing.py').exists() and (folder / '03_modelling.py').exists()

# Look for the uploaded bundle in common places.
bundle_candidates = [
    Path('/content/hf_risk_colab_bundle.zip'),
    Path.cwd() / 'hf_risk_colab_bundle.zip',
]
if Path('/content').exists():
    try:
        for p in Path('/content').rglob('hf_risk_colab_bundle.zip'):
            bundle_candidates.append(p)
            if len(bundle_candidates) >= 8:
                break
    except Exception:
        pass

bundle = next((b for b in bundle_candidates if b.exists()), None)
if bundle is not None:
    extract_root = Path('/content') if Path('/content').exists() else Path.cwd()
    with zipfile.ZipFile(bundle, 'r') as zf:
        zf.extractall(extract_root)
    print('Unzipped bundle:', bundle, '->', extract_root)

# Check common project locations first.
candidates = [
    Path.cwd(),
    Path.cwd() / 'MSc Project',
    Path('/content/MSc Project'),
    Path('/content/auto-annotated-portfolio-9bdeb/MSc Project'),
    Path('/content/auto-annotated-portfolio-9bdeb'),
    Path('/content/drive/MyDrive/MSc Project'),
    Path('/content/drive/MyDrive/auto-annotated-portfolio-9bdeb/MSc Project'),
]

project_dir = next((cand for cand in candidates if looks_like_project(cand)), None)

# Fallback: search for the script and accept its parent as project root.
if project_dir is None and Path('/content').exists():
    try:
        for hit in Path('/content').rglob('02_preprocessing.py'):
            cand = hit.parent
            if looks_like_project(cand):
                project_dir = cand
                break
    except Exception:
        pass

if project_dir is None:
    print('Current folder:', Path.cwd())
    if Path('/content').exists():
        print('/content sample:', [p.name for p in list(Path('/content').iterdir())[:20]])
    raise FileNotFoundError(
        'Could not find MSc Project scripts. Upload hf_risk_colab_bundle.zip to the Colab server '
        'or open this notebook from a workspace that already contains the project.'
)

os.chdir(project_dir)
print('Using project folder:', project_dir)
print('Files here:', [p.name for p in Path.cwd().iterdir() if p.name.endswith('.py')][:12])

Current folder: /content
/content sample: ['.config', 'sample_data']


FileNotFoundError: Could not find MSc Project scripts. Upload hf_risk_colab_bundle.zip to the Colab server or open this notebook from a workspace that already contains the project.

In [ ]:
# 3) Install dependencies into the Colab runtime.
# Runtime installs are temporary; rerun after reconnecting to a fresh Colab VM.
!python -m pip install -q -r requirements.txt

In [ ]:
# 4) Colab speed patch: use all available CPU cores on Colab/Linux.
# This reverses the local Windows workaround (n_jobs=1) in the runtime copy.
from pathlib import Path

for file_name in ['03_modelling.py', '04_survival.py']:
    path = Path(file_name)
    text = path.read_text(encoding='utf-8')
    updated = text.replace('n_jobs=1', 'n_jobs=-1')
    if updated != text:
        path.write_text(updated, encoding='utf-8')
        print(f'Updated {file_name}: n_jobs=1 -> n_jobs=-1')
    else:
        print(f'{file_name}: already using n_jobs=-1 or no n_jobs=1 found')

In [ ]:
# 5) Quick leakage check before heavy modelling.
from pathlib import Path

try:
    pd = __import__('pandas')
except Exception as exc:
    raise ImportError("pandas is required. Run cell 4 to install requirements first.") from exc

raw = pd.read_csv('data/dat.csv', nrows=1)
print('Raw dataset columns:', raw.shape[1])
print('Required data present:', Path('data/dat.csv').exists())
print('MIMIC data present:', Path('data/mimic_hf_cohort.csv').exists())
print('Script 10 present:', Path('10_advanced_evaluation.py').exists())

In [ ]:
# 6) Run the full clean pipeline.
# If a cell times out or disconnects, rerun from the failed script onward.
import os, subprocess, sys, time

scripts = [
    '02_preprocessing.py',
    '03_modelling.py',
    '04_survival.py',
    '05_shap.py',
    '08_external_validation_mimic.py',
    '10_advanced_evaluation.py',
]

env = os.environ.copy()
env['PYTHONIOENCODING'] = 'utf-8'

for script in scripts:
    print('\n' + '=' * 80)
    print(f'RUNNING: {script}')
    print('=' * 80)
    start = time.time()
    result = subprocess.run([sys.executable, script], env=env)
    mins = (time.time() - start) / 60
    print(f'FINISHED: {script} in {mins:.1f} min | exit code={result.returncode}')
    if result.returncode != 0:
        raise RuntimeError(f'{script} failed. Fix that script/output, then rerun this cell from the failed script onward.')

In [ ]:
# 7) Verification summary after pipeline completes.
import json
from pathlib import Path

try:
    pd = __import__('pandas')
except Exception as exc:
    raise ImportError("pandas is required. Run cell 4 to install requirements first.") from exc

bad_cols = [
    'inpatient.number',
    'dischargeDay',
    'Unnamed: 0',
    'outcome.during.hospitalization',
    're.admission.within.28.days',
    're.admission.within.3.months',
    're.admission.time..days.from.admission.',
    'return.to.emergency.department.within.6.months',
    'time.to.emergency.department.within.6.months',
]

x_test = pd.read_csv('outputs/X_test.csv')
present_bad = [c for c in bad_cols if c in x_test.columns]
print('X_test shape:', x_test.shape)
print('Leakage/ID/date columns still present:', present_bad or 'None')

with open('outputs/best_models_info.json', 'r', encoding='utf-8') as f:
    best = json.load(f)
print('Best model info:')
for k, v in best.items():
    print(f"  {k}: {v.get('name')} | test_auc={v.get('test_auc')}")

adv = Path('outputs/advanced_evaluation')
expected = [
    'calibration_curves.png', 'calibration_metrics.csv',
    'dca_curves.png', 'dca_results.csv',
    'subgroup_gender.png', 'subgroup_gender.csv',
    'subgroup_ckd.png', 'subgroup_ckd.csv',
    'advanced_evaluation_summary.json',
]
print('Advanced evaluation outputs:')
for name in expected:
    print(f'  {name}:', 'OK' if (adv / name).exists() else 'MISSING')

if Path('outputs/external_validation/external_validation_results.csv').exists():
    print('External validation:')
    display(pd.read_csv('outputs/external_validation/external_validation_results.csv'))

if Path('outputs/shap/shap_importance.csv').exists():
    print('Top SHAP features:')
    display(pd.read_csv('outputs/shap/shap_importance.csv').head(12))

## After It Finishes

Bring back or sync these folders/files to your local VS Code workspace if the extension does not automatically sync them:

- `MSc Project/models/`
- `MSc Project/outputs/`
- `MSc Project/03_modelling.py` and `04_survival.py` only if you intentionally want local files to stay at `n_jobs=-1`. For Windows local runs, keep `n_jobs=1`.